# RAG Basics: Loading PDF Documents

This notebook loads the tutorial arXiv papers with LangChain's `PyPDFLoader`. Each PDF page becomes a LangChain `Document` containing page text and source metadata.

In [1]:
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader


def find_project_root(start: Path) -> Path:
    """Find the nearest parent directory containing pyproject.toml."""
    for directory in (start, *start.parents):
        if (directory / "pyproject.toml").exists():
            return directory
    raise FileNotFoundError("Could not locate the project root")


project_root = find_project_root(Path.cwd().resolve())
pdf_directory = project_root / "data" / "raw" / "arxiv"
pdf_paths = sorted(pdf_directory.glob("*.pdf"))

if not pdf_paths:
    raise FileNotFoundError(
        f"No PDFs found in {pdf_directory}. Run "
        "`uv run python -m src.download_arxiv_papers` first."
    )

documents = []
for pdf_path in pdf_paths:
    loader = PyPDFLoader(pdf_path)
    documents.extend(loader.load())

print(f"Loaded {len(pdf_paths)} PDF files from {pdf_directory}")
print(f"Created {len(documents)} page-level documents")
print("Files:")
for pdf_path in pdf_paths:
    page_count = sum(
        document.metadata.get("source") == str(pdf_path)
        for document in documents
    )
    print(f"- {pdf_path.name}: {page_count} pages")

first_document = documents[0]
print("\nFirst document metadata:")
print(first_document.metadata)
print("\nFirst 500 characters:")
print(first_document.page_content[:500])

/var/folders/hf/0h4pd6zx1vv0dwjx8bgs9r_c0000gp/T/ipykernel_80341/609406098.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 5 PDF files from /Users/mlstudio/Claude-Code/ml-lab-new/tutorial-agentic-ai/data/raw/arxiv
Created 103 page-level documents
Files:
- 2606.20331.pdf: 32 pages
- 2606.20374.pdf: 17 pages
- 2606.20497.pdf: 27 pages
- 2606.20539.pdf: 6 pages
- 2606.20563.pdf: 21 pages

First document metadata:
{'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Robert Ganian; Mathis Rocton', 'doi': 'https://doi.org/10.48550/arXiv.2606.20331', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Computing Twin-Width via Treedepth and Vertex Integrity', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2606.20331v1', 'source': '/Users/mlstudio/Claude-Code/ml-lab-new/tutorial-agentic-ai/data/raw/arxiv/2606.20331.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1'}

First 500 characters:
Computing Twin-Width
via Treedept

## Split pages into retrieval chunks

Embedding complete pages often produces vectors that represent too many ideas at once. `RecursiveCharacterTextSplitter` creates smaller, overlapping chunks while preferentially splitting on paragraph, line, and word boundaries.

For this baseline, we use `chunk_size=1000` and `chunk_overlap=200`. These are practical starting values for academic text: chunks retain enough context for question answering, while the overlap reduces information loss at chunk boundaries. They should later be tuned using retrieval evaluation rather than treated as universal defaults.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(documents)
chunk_lengths = [len(chunk.page_content) for chunk in chunks]

print(f"Split {len(documents)} page documents into {len(chunks)} chunks")
print(f"Average chunk length: {sum(chunk_lengths) / len(chunk_lengths):.0f} characters")
print(f"Chunk length range: {min(chunk_lengths)}-{max(chunk_lengths)} characters")

sample_chunk = chunks[0]
print("\nSample chunk metadata:")
print(sample_chunk.metadata)
print("\nSample chunk text:")
print(sample_chunk.page_content[:1000])

Split 103 page documents into 445 chunks
Average chunk length: 869 characters
Chunk length range: 72-1000 characters

Sample chunk metadata:
{'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Robert Ganian; Mathis Rocton', 'doi': 'https://doi.org/10.48550/arXiv.2606.20331', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Computing Twin-Width via Treedepth and Vertex Integrity', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2606.20331v1', 'source': '/Users/mlstudio/Claude-Code/ml-lab-new/tutorial-agentic-ai/data/raw/arxiv/2606.20331.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1'}

Sample chunk text:
Computing Twin-Width
via Treedepth and Vertex Integrity
Robert Ganian/envel⌢pe/orcid
Algorithms and Complexity Group, TU Wien, Vienna, Austria
Mathis Rocton/envel⌢pe/orcid
Algorithms and Comple